# Health Sense AI — Stroke Risk Prediction Model
### End-to-End Clinical ML Pipeline & Exploratory Data Analysis

This notebook provides a complete walkthrough of the **Stroke Risk Prediction Module** for **Health Sense AI**.
It covers EDA, data cleaning, pipeline preprocessing, class imbalance benchmarking, Optuna tuning, probability calibration, SHAP explainability, and predictor API inference.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

# Set path relative to project root
sys.path.insert(0, str(Path.cwd().parent))
from src.utils import load_dataset
from src.preprocessing import clean_data, create_preprocessing_pipeline
from src.predictor import load_model, predict

print('All dependencies imported successfully!')

## STEP 1: Exploratory Data Analysis (EDA)
Examining dataset structure, column descriptions, missing values, duplicates, class distribution, and correlation.

In [ ]:
df_raw = load_dataset(Path('../data/healthcare-dataset-stroke-data.csv'))
print(f'Raw Dataset Shape: {df_raw.shape}')
display(df_raw.head())
print('\nData Schema Info:')
df_raw.info()
display(df_raw.describe())

In [ ]:
missing = df_raw.isnull().sum()
print('Missing Values:\n', missing[missing > 0])
dups = df_raw.duplicated(subset=[c for c in df_raw.columns if c != 'id']).sum()
print(f'Exact Duplicate Rows: {dups}')
print('\nStroke Target Ratio:')
print(df_raw['stroke'].value_counts())
print(df_raw['stroke'].value_counts(normalize=True).apply(lambda x: f'{x:.2%}'))

In [ ]:
from IPython.display import Image, display as display_img
fig_dir = Path('../outputs/figures')

eda_plots = ['missing_values.png', 'class_imbalance.png', 'eda_distributions.png', 'correlation_matrix.png']
for fig_name in eda_plots:
    p = fig_dir / fig_name
    if p.exists():
        print(f'=== Figure: {fig_name} ===')
        display_img(Image(filename=str(p)))

## STEP 2 & 3: Data Preprocessing & Pipeline Construction
Clean raw data, handle rare categories, apply `ColumnTransformer` with `SimpleImputer`, `StandardScaler`, `OneHotEncoder`, and domain `ClinicalFeatureAdder`.

In [ ]:
clean_df = clean_data(df_raw, is_training=True)
pipeline = create_preprocessing_pipeline()
X = clean_df.drop(columns=['stroke'])
y = clean_df['stroke'].values

X_trans = pipeline.fit_transform(X)
print(f'Clean Dataset Shape: {clean_df.shape}')
print(f'Transformed Matrix Shape: {X_trans.shape}')

## STEP 4-9: Evaluation, Calibration & Model Interpretation
Display ROC & PR curves, probability calibration reliability diagrams, and feature importance rankings.

In [ ]:
eval_plots = ['pr_roc_curves.png', 'calibration_curve.png', 'feature_importance.png']
for fig_name in eval_plots:
    p = fig_dir / fig_name
    if p.exists():
        print(f'=== Figure: {fig_name} ===')
        display_img(Image(filename=str(p)))

## STEP 10 & 13: Health Sense AI Predictor API Verification
Test `predict(patient)` API with an example patient payload.

In [ ]:
load_model(Path('../models'))

sample_patient = {
    'gender': 'Male',
    'age': 67,
    'hypertension': 1,
    'heart_disease': 0,
    'ever_married': 'Yes',
    'work_type': 'Private',
    'Residence_type': 'Urban',
    'avg_glucose_level': 182.4,
    'bmi': 31.2,
    'smoking_status': 'formerly smoked'
}

result = predict(sample_patient)
print(json.dumps(result, indent=4))